In [8]:
import json
import pandas as pd

with open("../reference/party_list_vote_ref.json", encoding="utf-8") as f:
    ref = json.load(f)

print(type(ref))
print(list(ref.keys()))
print()
print("results sample:")
print(ref["results"][:3])


<class 'dict'>
['province_name', 'constituency_number', 'type', 'summary_by_section', 'results', '_source_file', '_validation']

results sample:
[{'number': 1, 'party': 'ไทยทรัพย์ทวี', 'votes': 1480}, {'number': 2, 'party': 'เพื่อชาติไทย', 'votes': 473}, {'number': 3, 'party': 'ใหม่', 'votes': 373}]


In [9]:
# โหลด OCR data — concat ทั้ง 18 และ 17
df_18 = pd.read_csv("../output/final_csv/party_list.csv")   # ปรับ path ตามจริง
df_17 = pd.read_csv("../output/final_csv/17_party_list.csv")

# ดู columns ก่อน
print(df_18.columns.tolist())
print(df_17.columns.tolist())


['จังหวัด', 'เขตเลือกตั้งที่', 'ตำบล', 'อำเภอ', 'หน่วยเลือกตั้งที่', 'หมู่ที่', 'จำนวนผู้มีสิทธิเลือกตั้ง', 'จำนวนผู้มาแสดงตน', 'จำนวนบัตรที่ได้รับจัดสรร', 'จำนวนบัตรที่ใช้', 'จำนวนบัตรดี', 'จำนวนบัตรเสีย', 'จำนวนบัตรที่ไม่เลือกผู้สมัคร', 'จำนวนบัตรคงเหลือ', 'หมายเลข', 'พรรคการเมือง', 'คะแนน']
['จังหวัด', 'เขต', 'ชุดที่', 'บัตรดี', 'บัตรเสีย', 'บัตรที่ไม่เลือก', 'หมายเลข', 'พรรค', 'คะแนน']


In [10]:
# รวม OCR data
ocr_pl = pd.concat([df_18, df_17], ignore_index=True)

# รวมคะแนนต่อหมายเลขพรรค
ocr_total = ocr_pl.groupby("หมายเลข")["คะแนน"].sum().reset_index()
ocr_total.columns = ["number", "ocr_votes"]

# โหลด reference
ref_df = pd.DataFrame(ref["results"])

# Merge และเปรียบเทียบ
compare = ref_df.merge(ocr_total, on="number", how="left")
compare["diff"] = compare["ocr_votes"] - compare["votes"]
compare["diff_pct"] = (compare["diff"] / compare["votes"] * 100).round(2)

print(compare[["number", "party", "votes", "ocr_votes", "diff", "diff_pct"]].to_string())


    number                        party  votes  ocr_votes    diff  diff_pct
0        1                 ไทยทรัพย์ทวี   1480     1441.0   -39.0     -2.64
1        2                 เพื่อชาติไทย    473      465.0    -8.0     -1.69
2        3                         ใหม่    373      365.0    -8.0     -2.14
3        4                     มิติใหม่    216      213.0    -3.0     -1.39
4        5                     รวมใจไทย    264      249.0   -15.0     -5.68
5        6              รวมไทยสร้างชาติ   2000     1990.0   -10.0     -0.50
6        7                        พลวัต    101       99.0    -2.0     -1.98
7        8              ประชาธิปไตยใหม่    963      956.0    -7.0     -0.73
8        9                     เพื่อไทย   1360     1339.0   -21.0     -1.54
9       10                 ทางเลือกใหม่    315      301.0   -14.0     -4.44
10      11                     เศรษฐกิจ   1447     1428.0   -19.0     -1.31
11      12                   เสรีรวมไทย    277      270.0    -7.0     -2.53
12      13  

In [11]:
ocr_grand = ocr_pl["คะแนน"].sum()
ref_grand = ref_df["votes"].sum()

print(f"OCR รวม:      {ocr_grand:,}")
print(f"Reference รวม: {ref_grand:,}")
print(f"Diff:          {ocr_grand - ref_grand:,}")
print(f"Coverage:      {ocr_grand / ref_grand * 100:.2f}%")


OCR รวม:      92,814.0
Reference รวม: 94,553
Diff:          -1,739.0
Coverage:      98.16%
